# 02 - Diabetes Multistate Progression Model

## Overview

This notebook implements a multistate survival model for type 2 diabetes (T2D)
progression with the following state structure:

```
Prediabetes --> T2D --> Microvascular --> Macrovascular --> ESRD --> Death
     |           |           |                |              |        ^
     +-----------+-----------+----------------+--------------+--------+
                          (death as competing risk)
```

**Microvascular** complications: retinopathy, neuropathy, nephropathy (early).
**Macrovascular** complications: MI, stroke, peripheral vascular disease.
**ESRD**: End-stage renal disease (dialysis / transplant).

Key modelling feature: **HbA1c as a time-varying covariate** using Dynamic-DeepHit,
which can handle irregularly-spaced longitudinal measurements.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pathlib
import warnings
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import (
    concordance_index_ipcw,
    cumulative_dynamic_auc,
)
from sksurv.util import Surv

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = pathlib.Path("../data/synthea")
OUTPUT_DIR = pathlib.Path("../outputs/diabetes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load and preprocess synthetic diabetes cohort ───────────────────────────
patients = pd.read_csv(DATA_DIR / "patients.csv", parse_dates=["BIRTHDATE", "DEATHDATE"])
conditions = pd.read_csv(DATA_DIR / "conditions.csv", parse_dates=["START", "STOP"])
observations = pd.read_csv(DATA_DIR / "observations.csv", parse_dates=["DATE"])

# Diabetes-related SNOMED codes
DM_CODES = {
    "prediabetes": ["15777000"],
    "t2d": ["44054006"],
    "retinopathy": ["422034002", "232020009"],
    "neuropathy": ["230572002", "368581000119106"],
    "nephropathy": ["127013003", "431855005"],
    "mi": ["22298006"],
    "stroke": ["230690007"],
    "pvd": ["399957001"],
    "esrd": ["46177005", "236578006"],
    "ckd": ["431855005", "431856006", "433144002", "433146000"],
}
ALL_DM_CODES = [code for codes in DM_CODES.values() for code in codes]

# Filter cohort
dm_conditions = conditions[conditions["CODE"].astype(str).isin(ALL_DM_CODES)].copy()
dm_patient_ids = dm_conditions["PATIENT"].unique()
print(f"Patients with diabetes-related conditions: {len(dm_patient_ids)}")

# Build patient-level features
reference_date = pd.Timestamp("2024-01-01")
cohort = patients[patients["Id"].isin(dm_patient_ids)].copy()
cohort["age"] = (reference_date - cohort["BIRTHDATE"]).dt.days / 365.25
cohort["sex_male"] = (cohort["GENDER"] == "M").astype(int)
cohort["deceased"] = cohort["DEATHDATE"].notna().astype(int)
cohort["survival_years"] = np.where(
    cohort["deceased"] == 1,
    (cohort["DEATHDATE"] - cohort["BIRTHDATE"]).dt.days / 365.25,
    cohort["age"],
)

# Assign multistate labels
DM_STATES = OrderedDict([
    ("Prediabetes", 0),
    ("T2D", 1),
    ("Microvascular", 2),
    ("Macrovascular", 3),
    ("ESRD", 4),
    ("Death", 5),
])

MICRO_CODES = DM_CODES["retinopathy"] + DM_CODES["neuropathy"] + DM_CODES["nephropathy"]
MACRO_CODES = DM_CODES["mi"] + DM_CODES["stroke"] + DM_CODES["pvd"]
ESRD_CODES = DM_CODES["esrd"] + DM_CODES["ckd"]


def assign_dm_state(patient_id: str) -> str:
    pt = dm_conditions[dm_conditions["PATIENT"] == patient_id]
    codes = set(pt["CODE"].astype(str).values)
    if codes & set(ESRD_CODES):
        return "ESRD"
    if codes & set(MACRO_CODES):
        return "Macrovascular"
    if codes & set(MICRO_CODES):
        return "Microvascular"
    if codes & set(DM_CODES["t2d"]):
        return "T2D"
    return "Prediabetes"


cohort["dm_state"] = cohort["Id"].apply(assign_dm_state)
cohort["state_code"] = cohort["dm_state"].map(DM_STATES)
print("\nDiabetes state distribution:")
print(cohort["dm_state"].value_counts())

# Merge lab values
lab_codes = {
    "hba1c": "Hemoglobin A1c/Hemoglobin.total in Blood",
    "glucose": "Glucose",
    "bmi": "Body Mass Index",
    "sbp": "Systolic Blood Pressure",
    "egfr": "Estimated Glomerular Filtration Rate",
    "creatinine": "Creatinine",
    "total_chol": "Total Cholesterol",
    "ldl": "Low Density Lipoprotein Cholesterol",
    "triglycerides": "Triglycerides",
}

for feat_name, obs_desc in lab_codes.items():
    obs_feat = (
        observations[observations["DESCRIPTION"] == obs_desc]
        .sort_values("DATE")
        .groupby("PATIENT")["VALUE"]
        .last()
        .reset_index()
        .rename(columns={"PATIENT": "Id", "VALUE": feat_name})
    )
    obs_feat[feat_name] = pd.to_numeric(obs_feat[feat_name], errors="coerce")
    cohort = cohort.merge(obs_feat, on="Id", how="left")

FEATURE_COLS = ["age", "sex_male", "hba1c", "glucose", "bmi", "sbp",
                "egfr", "creatinine", "total_chol", "ldl", "triglycerides"]
TARGET_EVENT = "deceased"
TARGET_TIME = "survival_years"

for col in FEATURE_COLS:
    cohort[col] = cohort[col].fillna(cohort[col].median())

print(f"\nFinal cohort size: {len(cohort)}")
print(f"Event rate: {cohort[TARGET_EVENT].mean():.2%}")

In [ ]:
# ── Prepare HbA1c longitudinal data for Dynamic-DeepHit ────────────────────
# Extract all HbA1c measurements per patient over time
hba1c_long = (
    observations[
        (observations["DESCRIPTION"] == "Hemoglobin A1c/Hemoglobin.total in Blood")
        & (observations["PATIENT"].isin(dm_patient_ids))
    ][["PATIENT", "DATE", "VALUE"]]
    .copy()
)
hba1c_long["VALUE"] = pd.to_numeric(hba1c_long["VALUE"], errors="coerce")
hba1c_long = hba1c_long.dropna(subset=["VALUE"])
hba1c_long = hba1c_long.sort_values(["PATIENT", "DATE"])

# Compute time since first measurement (in years)
first_dates = hba1c_long.groupby("PATIENT")["DATE"].min().rename("first_hba1c_date")
hba1c_long = hba1c_long.merge(first_dates, on="PATIENT")
hba1c_long["time_years"] = (hba1c_long["DATE"] - hba1c_long["first_hba1c_date"]).dt.days / 365.25

print(f"Total HbA1c measurements: {len(hba1c_long)}")
print(f"Patients with HbA1c data: {hba1c_long['PATIENT'].nunique()}")
print(f"Median measurements per patient: {hba1c_long.groupby('PATIENT').size().median():.0f}")

# Visualise HbA1c trajectories for a sample of patients
fig, ax = plt.subplots(figsize=(12, 5))
sample_ids = hba1c_long["PATIENT"].unique()[:20]
for pid in sample_ids:
    pt_data = hba1c_long[hba1c_long["PATIENT"] == pid]
    ax.plot(pt_data["time_years"], pt_data["VALUE"], alpha=0.5, marker="o", ms=3)

ax.axhline(6.5, color="red", ls="--", label="T2D threshold (6.5%)")
ax.axhline(5.7, color="orange", ls="--", label="Prediabetes threshold (5.7%)")
ax.set_xlabel("Years since first HbA1c")
ax.set_ylabel("HbA1c (%)")
ax.set_title("HbA1c trajectories (sample of 20 patients)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Train/test split ────────────────────────────────────────────────────────
train_df, test_df = train_test_split(
    cohort, test_size=0.2, random_state=RANDOM_STATE, stratify=cohort[TARGET_EVENT]
)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS])
X_test = scaler.transform(test_df[FEATURE_COLS])
X_train_df = pd.DataFrame(X_train, columns=FEATURE_COLS, index=train_df.index)
X_test_df = pd.DataFrame(X_test, columns=FEATURE_COLS, index=test_df.index)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Train event rate: {train_df[TARGET_EVENT].mean():.2%}")
print(f"Test  event rate: {test_df[TARGET_EVENT].mean():.2%}")

In [ ]:
# ── Model 1: Cox PH baseline ────────────────────────────────────────────────
cox_df = X_train_df.copy()
cox_df["T"] = train_df[TARGET_TIME].values
cox_df["E"] = train_df[TARGET_EVENT].values

cph = CoxPHFitter(penalizer=0.01)
cph.fit(cox_df, duration_col="T", event_col="E")
cph.print_summary(columns=["coef", "exp(coef)", "p"])

cox_risk = -cph.predict_partial_hazard(
    X_test_df[FEATURE_COLS]
).values.ravel()
cox_cindex = concordance_index(
    test_df[TARGET_TIME].values, cox_risk, test_df[TARGET_EVENT].values
)
print(f"\nCox PH C-index: {cox_cindex:.4f}")

In [ ]:
# ── Model 2: Dynamic-DeepHit with time-varying HbA1c ───────────────────────
import torch
import torch.nn as nn


class DynamicDeepHit(nn.Module):
    """Simplified Dynamic-DeepHit: RNN encoder for longitudinal data +
    feedforward network for static features, combined for survival prediction.
    
    Reference: Lee et al., 'Dynamic-DeepHit: A Deep Learning Approach for
    Dynamic Survival Analysis With Competing Risks', IEEE TBME 2020.
    """

    def __init__(self, n_static: int, n_longitudinal: int, hidden_rnn: int = 32,
                 hidden_fc: int = 64, n_time_bins: int = 50, n_risks: int = 2,
                 dropout: float = 0.3):
        super().__init__()
        self.n_time_bins = n_time_bins
        self.n_risks = n_risks

        # RNN for longitudinal features (e.g., HbA1c over time)
        self.rnn = nn.GRU(
            input_size=n_longitudinal,
            hidden_size=hidden_rnn,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
        )

        # Feedforward for static features
        self.static_net = nn.Sequential(
            nn.Linear(n_static, hidden_fc),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fc, hidden_fc // 2),
            nn.ReLU(),
        )

        # Combined output: predict PMF over (time_bin, risk_type)
        combined_dim = hidden_rnn + hidden_fc // 2
        self.output_net = nn.Sequential(
            nn.Linear(combined_dim, hidden_fc),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fc, n_time_bins * n_risks),
        )

    def forward(self, x_static: torch.Tensor, x_long: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x_static: (batch, n_static)
            x_long: (batch, seq_len, n_longitudinal)
        Returns:
            pmf: (batch, n_time_bins, n_risks) - predicted probability mass
        """
        # RNN encoding
        _, h_n = self.rnn(x_long)  # h_n: (num_layers, batch, hidden_rnn)
        rnn_out = h_n[-1]  # last layer hidden state

        # Static encoding
        static_out = self.static_net(x_static)

        # Combine
        combined = torch.cat([rnn_out, static_out], dim=1)
        logits = self.output_net(combined)
        logits = logits.view(-1, self.n_time_bins, self.n_risks)

        # Softmax across time and risk to get valid PMF
        pmf = torch.softmax(logits.view(-1, self.n_time_bins * self.n_risks), dim=1)
        pmf = pmf.view(-1, self.n_time_bins, self.n_risks)
        return pmf


# Prepare longitudinal data: pad HbA1c sequences
MAX_SEQ_LEN = 20  # max number of HbA1c measurements to use


def build_longitudinal_tensor(patient_ids, hba1c_df, max_len=MAX_SEQ_LEN):
    """Build padded tensor of HbA1c sequences."""
    sequences = []
    for pid in patient_ids:
        pt = hba1c_df[hba1c_df["PATIENT"] == pid][["time_years", "VALUE"]].values
        if len(pt) == 0:
            pt = np.zeros((1, 2))  # zero-pad if no data
        if len(pt) > max_len:
            pt = pt[-max_len:]  # take most recent
        padded = np.zeros((max_len, 2))
        padded[-len(pt):] = pt
        sequences.append(padded)
    return torch.FloatTensor(np.array(sequences))


X_long_train = build_longitudinal_tensor(train_df["Id"].values, hba1c_long)
X_long_test = build_longitudinal_tensor(test_df["Id"].values, hba1c_long)
X_static_train = torch.FloatTensor(X_train)
X_static_test = torch.FloatTensor(X_test)

print(f"Longitudinal tensor shape (train): {X_long_train.shape}")
print(f"Static tensor shape (train):       {X_static_train.shape}")

# Discretise time for Dynamic-DeepHit
N_TIME_BINS = 50
time_cuts = np.linspace(0, train_df[TARGET_TIME].max(), N_TIME_BINS + 1)
train_time_bin = np.digitize(train_df[TARGET_TIME].values, time_cuts) - 1
train_time_bin = np.clip(train_time_bin, 0, N_TIME_BINS - 1)

# Competing risk labels: 0=censored, 1=DM-related death, 2=other death
train_cr = np.where(
    train_df["deceased"] == 1,
    np.where(train_df["dm_state"].isin(["ESRD", "Macrovascular"]), 1, 2),
    0,
)

# Train
model_ddh = DynamicDeepHit(
    n_static=len(FEATURE_COLS), n_longitudinal=2,
    hidden_rnn=32, hidden_fc=64, n_time_bins=N_TIME_BINS, n_risks=2,
)
optimizer = torch.optim.Adam(model_ddh.parameters(), lr=1e-3, weight_decay=1e-4)

n_epochs = 80
losses = []

model_ddh.train()
for epoch in range(n_epochs):
    optimizer.zero_grad()
    pmf = model_ddh(X_static_train, X_long_train)

    # Negative log-likelihood loss
    loss = torch.tensor(0.0)
    for i in range(len(train_df)):
        t_bin = train_time_bin[i]
        event = train_cr[i]
        if event > 0:
            # Log-likelihood of event at observed time and risk
            loss -= torch.log(pmf[i, t_bin, event - 1] + 1e-8)
        else:
            # Censored: survival beyond observed time
            surv = 1.0 - pmf[i, :t_bin + 1, :].sum()
            loss -= torch.log(surv + 1e-8)
    loss /= len(train_df)

    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, lw=1.5)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Dynamic-DeepHit training loss")
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluation ──────────────────────────────────────────────────────────────
# Cox PH evaluation
y_train_surv = Surv.from_arrays(
    event=train_df[TARGET_EVENT].astype(bool).values,
    time=train_df[TARGET_TIME].values,
)
y_test_surv = Surv.from_arrays(
    event=test_df[TARGET_EVENT].astype(bool).values,
    time=test_df[TARGET_TIME].values,
)

time_horizons = [1.0, 3.0, 5.0]
max_time = test_df[TARGET_TIME].max()
valid_horizons = [t for t in time_horizons if t < max_time]

# Dynamic-DeepHit: compute risk score as CIF at median time
model_ddh.eval()
with torch.no_grad():
    pmf_test = model_ddh(X_static_test, X_long_test)
    # CIF for risk 1 (DM-related death) at median time bin
    median_bin = N_TIME_BINS // 2
    ddh_risk = pmf_test[:, :median_bin, 0].sum(dim=1).numpy()

ddh_cindex = concordance_index(
    test_df[TARGET_TIME].values, -ddh_risk, test_df[TARGET_EVENT].values
)

# Collect results
results = {"Model": [], "C-index": []}
for t in valid_horizons:
    results[f"AUC@{int(t)}y"] = []

for model_name, risk_scores, cindex in [
    ("Cox PH", cox_risk, cox_cindex),
    ("Dynamic-DeepHit", ddh_risk, ddh_cindex),
]:
    results["Model"].append(model_name)
    results["C-index"].append(cindex)
    try:
        auc_vals, _ = cumulative_dynamic_auc(
            y_train_surv, y_test_surv, risk_scores, valid_horizons
        )
        for i, t in enumerate(valid_horizons):
            results[f"AUC@{int(t)}y"].append(auc_vals[i])
    except Exception:
        for t in valid_horizons:
            results[f"AUC@{int(t)}y"].append(np.nan)

results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("MODEL COMPARISON (Diabetes Track)")
print("=" * 60)
print(results_df.to_string(index=False))

# Calibration: KM by risk quartile
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, risk) in zip(axes, [("Cox PH", cox_risk), ("Dynamic-DeepHit", ddh_risk)]):
    test_tmp = test_df.copy()
    test_tmp["rq"] = pd.qcut(-risk, 4, labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"])
    kmf = KaplanMeierFitter()
    for q in ["Q1 (low)", "Q2", "Q3", "Q4 (high)"]:
        mask = test_tmp["rq"] == q
        if mask.sum() > 0:
            kmf.fit(test_tmp.loc[mask, TARGET_TIME],
                    event_observed=test_tmp.loc[mask, TARGET_EVENT], label=q)
            kmf.plot_survival_function(ax=ax)
    ax.set_title(f"{name}: KM by risk quartile")
    ax.set_xlabel("Time (years)")
    ax.set_ylabel("Survival")

plt.tight_layout()
plt.show()

In [ ]:
# ── Generate model card ─────────────────────────────────────────────────────
model_card = f"""# Model Card: Diabetes Multistate Progression Model

## Model Details
- **Model type**: Multistate survival model (Cox PH, Dynamic-DeepHit)
- **Version**: 0.1.0 (development)
- **Training date**: {pd.Timestamp.now().strftime('%Y-%m-%d')}
- **Framework**: lifelines, PyTorch
- **Time-varying covariate**: HbA1c (longitudinal)

## State structure
Prediabetes -> T2D -> Microvascular -> Macrovascular -> ESRD -> Death

## Performance Summary
{results_df.to_markdown(index=False)}

## Intended Use
Research prototype for DACH insurance underwriting support.
NOT approved for production use or clinical decision-making.

## Limitations
- Trained on synthetic (Synthea) data
- HbA1c trajectory modelling depends on measurement frequency
- No oral glucose tolerance test or C-peptide data
- US prescribing patterns may differ from DACH
"""

card_path = pathlib.Path("../governance/model_card_diabetes.md")
card_path.parent.mkdir(parents=True, exist_ok=True)
card_path.write_text(model_card)
print(f"Model card saved to {card_path}")